# E2 - Large TinyFat anatomy-target queue preparation


## Notebook Setup
This cell locates the project root and imports the generic modules used for sampling, manifest generation, queue inspection, GUI review, and sync command generation.


In [ ]:
from pathlib import Path
from typing import Optional
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display


def find_project_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        script = candidate / "scripts" / "e2_tinyfat_generate_validate.py"
        if script.exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the project root from the current notebook working directory.")


PROJECT_ROOT = find_project_root()
HELPER_DIR = PROJECT_ROOT / "experiments" / "master-thesis" / "notebook_helpers"
for path in (PROJECT_ROOT, HELPER_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"HELPER_DIR={HELPER_DIR}")
print(f"pandas={pd.__version__}")


## Experiment Overview
E2 samples 1000 anatomies and creates a TinyFat queue for one selected E1-style config. Targets are generated per non-aorta branch using configurable positions such as `start`, `mid`, `end`, or `random`.


## Experiment Setup


### E2.1 Sample random anatomies

Edit `E2_SAMPLE_N` and `E2_SAMPLE_SEED` if needed. Default is `n=1000`.

In [ ]:
# E2.1 Sample random anatomies
import subprocess

E2_SAMPLE_N = 1000
E2_SAMPLE_SEED = 2026
E2_SAMPLE_JSON = PROJECT_ROOT / "results" / "experimental_prep" / "sample_1000_e2.json"

cmd = [
    sys.executable,
    "scripts/sample_e2_anatomies.py",
    "--n", str(E2_SAMPLE_N),
    "--seed", str(E2_SAMPLE_SEED),
    "--output", str(E2_SAMPLE_JSON),
    "--workers", "8",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
print(f"E2_SAMPLE_JSON={E2_SAMPLE_JSON}")


### E2.2 Generate TinyFat queue

Set `E2_CONFIG_ID` to exactly one config. `E2_NUM_BATCHES` is the number of trials per candidate for every anatomy-target setup; default is `50`. Friction is fixed to `0.1` unless you intentionally override it.

Target placement is configurable per non-aorta branch. Set `E2_TARGETS_PER_BRANCH` and give the same number of comma-separated `E2_TARGET_POSITIONS` entries. Supported positions are `start`, `mid`, `end`, and `random`, e.g. `"end,random,random"`. Random positions are resolved deterministically from `E2_TARGET_SEED_START` and stored in `targets.json`.


In [ ]:
# E2.2 Generate TinyFat queue manifest and sbatch script
E2_CONFIG_ID = 2          # 1 deterministic/fixed, 2 deterministic/random, 3 stochastic/fixed, 4 stochastic/random
E2_NUM_BATCHES = 50       # trials per candidate per anatomy-target setup
E2_MAX_EPISODE_STEPS = 1000
E2_FRICTION = 0.1
E2_TARGETS_PER_BRANCH = 1
E2_TARGET_POSITIONS = "end"  # examples: "start,mid,end" or "end,random,random"
E2_TARGET_SEED_START = 9000
E2_OUTPUT_ROOT = PROJECT_ROOT / "results" / "master_thesis" / "e2_tinyfat"

cmd = [
    sys.executable,
    "scripts/e2_tinyfat_generate_validate.py",
    "--sample-json", str(E2_SAMPLE_JSON),
    "--output-root", str(E2_OUTPUT_ROOT),
    "--config-id", str(E2_CONFIG_ID),
    "--num-batches", str(E2_NUM_BATCHES),
    "--max-episode-steps", str(E2_MAX_EPISODE_STEPS),
    "--friction", str(E2_FRICTION),
    "--targets-per-branch", str(E2_TARGETS_PER_BRANCH),
    "--target-positions", E2_TARGET_POSITIONS,
    "--target-seed-start", str(E2_TARGET_SEED_START),
    #"--probe-cluster",
    "--no-write-trace",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
print(f"E2_OUTPUT_ROOT={E2_OUTPUT_ROOT}")
print("Generated sbatch script: scripts/e2_tinyfat_work.sbatch")


## Analysis And Review


### Load Generated Queue Into DataFrames
Inspect the manifest and target metadata before syncing or submitting the cluster job.


### E2.3 Inspect generated queue

This confirms how many anatomies, targets, and jobs were materialized before syncing to the cluster.

In [ ]:
# E2.3 Inspect generated queue
from pathlib import Path
import json
import pandas as pd

manifest_path = E2_OUTPUT_ROOT / "metadata" / "job_manifest.json"
targets_path = E2_OUTPUT_ROOT / "metadata" / "targets.json"
manifest = json.loads(manifest_path.read_text())
targets = json.loads(targets_path.read_text())

jobs = pd.DataFrame(manifest["jobs"])
print(f"manifest={manifest_path}")
print(f"targets={targets_path}")
print(f"n_anatomies={len(targets['selected_anatomies'])}")
print(f"n_jobs={len(jobs)}")
print(f"config_id={manifest['config_id']}")
print(f"trial_count={manifest['trial_count']}")
print(f"friction={manifest['friction']}")

target_counts = [len(row["targets"]) for row in targets["selected_anatomies"]]
display(pd.Series(target_counts, name="targets_per_anatomy").describe())
display(jobs[["config_id", "anatomy_id", "target_index", "seed_base", "partition", "worker_count", "output_dir"]].head(20))


### E2.4 Eval V2-style mesh + target review GUI

Launch a separate Qt/PyVista GUI using the same visualization stack as the Eval V2 target selection screen. The GUI loads one anatomy at a time, renders the vessel mesh, branch centerlines, and all generated targets. Use mouse interaction in the 3D view, `Next` / `Prev`, or arrow keys to step through the selected anatomies. Review decisions are saved to JSON.


In [ ]:
# E2.4 Launch Eval V2-style mesh + target review GUI
import os
import subprocess

E2_TARGETS_JSON = E2_OUTPUT_ROOT / "metadata" / "targets.json"
E2_REVIEW_JSON = PROJECT_ROOT / "results" / "master_thesis" / "e2_target_review" / "e2_target_review_gui.json"

if not E2_TARGETS_JSON.exists():
    display(Markdown(f"No E2 targets found yet: `{E2_TARGETS_JSON}`. Run E2.2 first."))
else:
    cmd = [
        sys.executable,
        "scripts/e2_target_review_gui.py",
        "--targets-json", str(E2_TARGETS_JSON),
        "--review-json", str(E2_REVIEW_JSON),
    ]
    print(" ".join(cmd))
    env = os.environ.copy()
    env.pop("QT_PLUGIN_PATH", None)
    conda_platforms = Path(sys.prefix) / "plugins" / "platforms"
    if conda_platforms.exists():
        env["QT_QPA_PLATFORM_PLUGIN_PATH"] = str(conda_platforms)
    proc = subprocess.Popen(cmd, cwd=PROJECT_ROOT, env=env)
    print(f"Started GUI pid={proc.pid}")
    print(f"Review decisions will be saved to: {E2_REVIEW_JSON}")


### E2.5 Sync and submit


In [ ]:
# E2.5 Commands to sync and submit E2 to TinyFat
remote_root = "/home/woody/iwhr/iwhr106h/master-project"
print(f"""
rsync -av {E2_OUTPUT_ROOT} \\
  iwhr106h@tinyx.nhr.fau.de:{remote_root}/results/master_thesis/

rsync -av scripts/e2_tinyfat_work.sbatch experiments/master-thesis/run_e2_cell.py \\
  scripts/e2_tinyfat_generate_validate.py scripts/sample_e2_anatomies.py \\
  experiments/master-thesis/notebook_helpers/e2_tinyfat.py \\
  iwhr106h@tinyx.nhr.fau.de:{remote_root}/

ssh iwhr106h@tinyx.nhr.fau.de
cd {remote_root}

# If generated locally, rewrite local absolute paths to cluster absolute paths:
for f in results/master_thesis/e2_tinyfat/metadata/*.json scripts/e2_tinyfat_work.sbatch; do
  sed -i 's#{PROJECT_ROOT}#{remote_root}#g' "$f"
done

sbatch.tinyfat scripts/e2_tinyfat_work.sbatch
""")
